In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import torch
import torchvision
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from pathlib import Path

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

os.makedirs("outputs", exist_ok=True)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# The standard device check — you'll use this pattern in every PyTorch notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version:     {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")

## PyTorch Tensors
A PyTorch tensor is the fundamental data type for everything in deep learning — model weights, input images, predictions, and loss values are all tensors. If NumPy arrays feel familiar at this point, tensors will too. They support the same slicing, shapes, and arithmetic. The key difference is that tensors can live on a GPU, which is what makes the massively parallel computation behind neural networks practical.

### Tensor Question 1
Create the following tensors and, for each one, print its value, shape, dtype, and device.  What device are these tensors on right now? If you were running a training loop on the GPU, why would it matter that your model weights and your input tensors are on the same device?

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

b = torch.zeros(2, 3)
c = torch.ones(4)

# Finding value, shape, dtype, and device of each tensor
print(f"Value : {a}")
print(f"Shape : {a.shape}")
print(f"dtype : {a.dtype}")
print(f"Device : {a.device}")

print(f"Value : {b}")
print(f"Shape : {b.shape}")
print(f"dtype : {b.dtype}")
print(f"Device : {b.device}")

print(f"Value : {c}")
print(f"Shape : {c.shape}")
print(f"dtype : {c.dtype}")
print(f"Device : {c.device}")

# The tensors are all on CPU.  It matters that they are on the same device because we cannot perform operations between tensors on different devices.  The memory of CPU and GPU are sepeate and do not automatically share data.

### Tensor Question 2

1. Compute and print the element-wise square root using torch.sqrt().
2. Compute and print the sum using .sum().
3. Compute and print the mean using .mean().
4. Find and print the index of the maximum value using .argmax().

.argmax() appears in nearly every inference example you'll encounter. In the context of a classifier that outputs scores for 1,000 classes, what does .argmax() give you?

In [ ]:
x = torch.tensor([1.0, 4.0, 9.0, 16.0, 25.0])

x_sqrt = torch.sqrt(x)
x_sum = torch.sum(x)
x_mean = torch.mean(x)
# argmax returns index of the largest value
x_argmax = torch.argmax(x)

print(f"Square root : {x_sqrt}")
print(f"Sum : {x_sum}")
print(f"Mean : {x_mean}")
print(f"Argument (Index) of Maximum : {x_argmax}")

# Because argmax() returns the index of the class with the highest score.  Using it on a classifier that gives output scores for 1,000 classes, it will return the index of the class that has the highest probability/confidence.

### Tensor Question 3
Move tensor a from Question 1 to the GPU, then bring it back to CPU and convert it to a NumPy array.

Why does PyTorch require .cpu() before you can call .numpy()? What does this tell you about where NumPy arrays live?

In [ ]:
a_gpu   = a.to(device)
print(f"a_gpu device: {a_gpu.device}")

a_back  = a_gpu.cpu()
a_numpy = a_back.numpy()
print(f"numpy type: {type(a_numpy)}")
print(f"numpy values:\n{a_numpy}")

# Pytorch requires cpu() before we can call numpy() because numpy() only works on CPU and a was on cuda because we had just moved it.

### Tensor Question 4
1. Reshape t to (4, 6) and print the shape.
2. Reshape t to (2, 3, 4) and print the shape.
3. Take your result from step 1 and add a new dimension at position 0. Print the new shape.

A single image tensor typically has shape (channels, height, width). Neural networks expect batches with shape (batch_size, channels, height, width). What operation accomplishes this when you are processing one image at a time, and why does it matter?

In [ ]:
t = torch.arange(24).float()

t_reshape_1 = t.reshape(4, 6)
print(f"Reshape to (4, 6) : {t_reshape_1}")

t_reshape_2 = t.reshape(2, 3, 4)
print(f"Reshape to (2, 3, 4) : {t_reshape_2}")

t_unsqueeze_1 = t_reshape_1.unsqueeze(0)
# equivalent : t[None, :] Adds dimension to front
print(f"Add dimension to position 0 : {t_unsqueeze_1}")

# The unsqueeze operation accomplishes adding a dimension to the image, even when processing one image at a time.  It matters because neural networks expects 4D input even on a single image (will still expect it in a folder)

### Tensor Question 5
1. Compute the matrix product using NumPy and print the result.
2. Compute the same product using PyTorch and print the result.
3. Confirm the outputs match.

At a high level, what role does matrix multiplication play as data passes through a single layer of a neural network?

In [ ]:
np_a = np.array([[1.0, 2.0], [3.0, 4.0]])
np_b = np.array([[5.0, 6.0], [7.0, 8.0]])

t_a  = torch.tensor(np_a, dtype=torch.float32)
t_b  = torch.tensor(np_b, dtype=torch.float32)

# @ is recommended for both np and pytorch, but both can use matmul(a, b).  Used both just to see outcome
np_matrix_product = np_a @ np_b
t_matrix_product = torch.matmul(t_a, t_b)

print(f"Numpy Matrix Product :\n{np_matrix_product}")
print(f"Pytorch Matrix Product :\n{t_matrix_product}")

# Matrix multiplication is a mechanism that lets a neural network layer combine and transform input features into new representations at a high level.

## Pretrained Models
Training a CNN from scratch requires millions of labeled images and days of GPU time. Pretrained models skip all of that — TorchVision ships with CNNs that were already trained on ImageNet, a dataset of over one million images across 1,000 categories. These models have learned to recognize edges, textures, shapes, and high-level objects. Loading one takes two lines of code. In practice, pretrained models are the default starting point for almost every real-world computer vision project.

### Model Question 1
ResNet18 has roughly 11 million parameters. Training it from scratch required approximately 1.2 million labeled ImageNet images and days of multi-GPU compute. What does that tell you about the practical value of starting from pretrained weights when you're on a deadline or a budget?

In [ ]:
weights = ResNet18_Weights.DEFAULT
model   = models.resnet18(weights=weights)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# The practical use of starting from pretrained weights is time.  It would take days to compute and there is no guarantee by the deadline it would work, or that it would stay within budget training all those images.

### Model Question 2
Print the full model architecture by running print(model). Read through the output — it shows every layer and its configuration.

1. What is the name of the final layer in ResNet18, and what is its output size? (This number is the total count of ImageNet categories the model can predict.)
2. Can you identify the blocks named layer1 through layer4? These are the "deep" part of the network — the feature extractor. In plain terms, what does it mean for a network to be "deep"?

In [ ]:
print(model)

# (fc): Linear(in_features=512, out_features=1000, bias=True)
# Final Layer : fc
# Output Size : 1000

# A deep network refers to a neural network with multiple hidden layers between the input and output layers.  It is designed to learn complex, non-linear representations.  In plain terms it has many layers stacked on top of each other.

### Model Question 3
1. What does .to(device) do, and why does it need to match the device your input tensors will be on?
2. What does model.eval() change about the model's behavior? Name at least one layer type that behaves differently in training mode vs. evaluation mode.

In [ ]:
model = model.to(device)
model.eval()
print("Model ready for inference.")

# .to(device) moves the model to the the device specified in the model's parameter.  It matters because the model and input have to be on the same device.  CPU and GPU do not share memory and cannot directly interact.
# model.eval() switches the model into evaluation (inference) mode.  It changes the behavior of certain layers but not weights.  One layer type that behaves differently in training mode vs evaluation mode is batch normalization.  Training uses batch statistics and updates running averages while evaluation only uses stored running averages.

### Model Question 4
1. What does the resize/crop step accomplish?
2. What does ToTensor() do to the pixel value range?
3. What is normalization doing, and why does it use ImageNet's specific mean and standard deviation values rather than, say, mean=0.5, std=0.5?

In [ ]:
preprocess = weights.transforms()
print(preprocess)

# Resize/crop ensures all the images are a uniform size and aspect ratio preparing them for neural network input.
# ToTensor() converts an image into a floating-point tensor and scales the pixel values into a specific range.
# Normalization standardizes pixel values so that it looks the same to the model as what it was trained on.  ResNet was trained on ImageNet with specific pixel statistics (each channel has a typical spread of values and color channel has it's own brightness), so normalization uses ImageNet's specific mean and standard deviation.

## Running Inference
With a model loaded and preprocessing defined, running inference on a new image takes about five lines of code. This section walks you through each step using real images from the Intel Image Classification dataset.

In [ ]:
import random
random.seed(42)

DATA_DIR = Path("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test")
LABELS   = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

def load_sample_image(label):
    """Load a random image file from the given class folder."""
    class_dir = DATA_DIR / label
    img_path  = random.choice(list(class_dir.glob("*.jpg")))
    return Image.open(img_path).convert("RGB"), img_path.name

Get the ImageNet class labels from the weights metadata — no separate download needed.

In [ ]:
imagenet_classes = weights.meta["categories"]
print(f"Number of classes: {len(imagenet_classes)}")
print(f"First 5 labels: {imagenet_classes[:5]}")

### Inference Question 1
Write a function that runs inference on a single PIL image and returns the top-5 predicted class names and their probabilities. The function signature and steps are given below.

Does the top prediction make sense? Remember that the model was trained on ImageNet's 1,000 categories, which include things like "alp", "valley", and "lakeside" rather than simply "mountain". Do any of the top-5 labels map onto what you'd describe as a mountain scene?

In [ ]:
def get_top5_predictions(model, preprocess, image, device, class_labels):
    """
    Run inference on a PIL image and return the top-5 predictions.
    Returns a list of (class_name, probability) tuples.
    """
    # Step 1: Preprocess the image and add a batch dimension
    # (hint: use preprocess(), .unsqueeze(0), and .to(device))
    processed_image = preprocess(image).unsqueeze(0).to(device)

    # Step 2: Run inference inside a torch.no_grad() block
    # (hint: call model() on your input tensor to get output of shape (1, 1000))
    with torch.no_grad():
        output = model(processed_image)

    # Step 3: Convert raw scores (logits) to probabilities
    # (hint: use torch.nn.functional.softmax on output[0])
    probabilities = torch.nn.functional.softmax(output[0], dim = 0)

    # Step 4: Get the top 5 predictions using torch.topk
    # (hint: returns top_probs and top_indices)
    top_probs, top_indices = torch.topk(probabilities, 5)
    
    # Step 5: Build and return a list of (class_name, probability) tuples
    results = []
    for probability, idx in zip(top_probs, top_indices):
        class_name = class_labels[idx]
        results.append((class_name, probability.item()))
        
    return results

In [ ]:
# Test it on one mountain image
img, img_name = load_sample_image("mountain/")
preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)

print(f"\nTop-5 predictions for '{img_name}':")
for class_name, prob in preds:
    print(f"  {class_name:30s}  {prob:.4f}")

# The top prediction makes sense as a mountain.  There is only one other prediction that made a little sense, a Volcano, but even that is not under the Mountain label.

### Inference Question 2
Run inference on one image from each of the six scene classes. For each, print the top-3 predictions.

Which classes does the model seem most confident about (high top-1 probability)? Which does it seem least confident about? Is there a pattern?

In [ ]:
for label in LABELS:
    img, img_name = load_sample_image(label)
    preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)[:3]
    print(f"\n[{label}]  {img_name}")
    for class_name, prob in preds:
        print(f"  {class_name:30s}  {prob:.4f}")

# The model is most confident about predicting if an image is a mountain.  The least confidence it seems to have is for buildings and streets.  The pattern of the least confident seem to be where they can be very different.  A building can be a small shack, but it can also be a 100 story sky scraper.  The same for streets, one can be paved with urban background (buildings which would cause overlap) or an unpaved dirt road.

### Inference Question 3
The raw output of the model before softmax is called logits — unconstrained scores that can be any real number. After softmax they become probabilities that sum to 1. Observe the difference.

Why do neural networks output logits internally rather than probabilities? In a production pipeline that needs to filter out low-confidence predictions, which representation would you work with — logits or probabilities — and why?

In [ ]:
img, _ = load_sample_image("forest")
input_tensor = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)

probs = torch.nn.functional.softmax(logits[0], dim=0)

print(f"Logit  range: min={logits.min():.2f}, max={logits.max():.2f}")
print(f"Prob   range: min={probs.min():.6f}, max={probs.max():.4f}")
print(f"Probs sum to: {probs.sum():.6f}")
print(f"Top prediction: {imagenet_classes[probs.argmax().item()]}  ({probs.max():.4f})")

# Neural networks use logits because they are more useful mathematically and numerically stable for learning. It would be better for me as a human to work with probabilities.  Probabilities are fixed with a sum of 1 and can not be below 0.  It is also consistent accross all models and can read the confidence from it.  Less than 0.5 would tell me to reject the prediction.

### Inference Question 4
Create a visualization that shows an image alongside a horizontal bar chart of its top-5 predictions. Use plt.subplots(1, 2) with one panel for the image and one for the bar chart. Save it to outputs/warmup_inference_viz.png.

You have all the pieces — img, preds, and plt — from the previous questions. Write the visualization yourself.

How would you adapt this kind of visualization for a dashboard that a non-technical team member needs to review flagged predictions? What threshold on the top-1 probability might you use to decide when a prediction is "confident enough" to act on?

In [ ]:
for label in LABELS:
    img, img_name = load_sample_image(label)
    preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)

# Unpacking preds so have a list of names and predictions
names = [name for name, prob in preds]
probs = [prob for name, prob in preds]

fig, ax = plt.subplots(1, 2)

ax[0].imshow(img)
ax[0].set_title(img_name)
ax[0].axis("off")

ax[1].barh(names, probs)
ax[1].set_title("Top 5 Predictions")
ax[1].set_xlabel("Probability")
ax[1].set_ylabel("Prediction")
ax[1].invert_yaxis()

plt.tight_layout()
plt.savefig("outputs/warmup_inference_viz.png")
plt.close()